In [1]:
import os
import numpy as np
# 设置 Hugging Face 镜像（如需要）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from vllm import LLM, SamplingParams

# ============================================================
# 第一步：准备知识库文档
# ============================================================
from charset_normalizer import from_path

def load_txt_lines(file_path):
    result = from_path(file_path).best()
    enc = result.encoding if result else None

    if enc:
        try:
            print(f"识别编码{enc}")
            with open(file_path, 'r', encoding=enc) as f:
                documents = f.readlines()
            return documents
        except UnicodeDecodeError:
            raise OSError(f"识别编码{enc}错误，文件解码失败")
    else:
        raise OSError("无法确定文件编码")

file_path = './data/胖虎大战高达拯救比奇堡.txt'
documents = load_txt_lines(file_path)

INFO 07-24 10:31:26 __init__.py:190] Automatically detected platform cuda.
识别编码utf_8


In [2]:
page_doc = []
maxlen = 0
for raw in documents[1:]:
    maxlen = max(maxlen, len(raw))  # 更新最大句子长度
    page_doc.append(raw.strip())  # 去除首尾空格
print(f"文档总句子数: {len(page_doc)}")
print(f"最大句子长度: {maxlen}")
del documents

文档总句子数: 31
最大句子长度: 163


In [3]:

# ============================================================
# 第二步：加载嵌入模型，为文档生成向量
# ============================================================
print("正在加载嵌入模型 intfloat/e5-small ...")
# vLLM 0.7.2 用 task="embed" 来加载嵌入模型
try:
    embedding_llm = LLM(
        model="intfloat/e5-small",
        task="embed",              # 指定嵌入任务
        enforce_eager=True
    )
    print("嵌入模型加载成功")
except Exception as e:
    print(f"加载嵌入模型失败: {e}")
    print("尝试降级方案：使用 sentence-transformers ...")
    # 降级方案：使用 sentence-transformers
    from sentence_transformers import SentenceTransformer
    # 创建降级方案的替代函数（见下方）

正在加载嵌入模型 intfloat/e5-small ...
INFO 07-24 10:31:30 config.py:2382] Downcasting torch.float32 to torch.float16.
WARNING 07-24 10:31:38 cuda.py:95] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 07-24 10:31:38 config.py:678] Async output processing is not supported on the current platform type cuda.
INFO 07-24 10:31:39 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='intfloat/e5-small', speculative_config=None, tokenizer='intfloat/e5-small', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=512, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_dec

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 07-24 10:31:43 model_runner.py:1115] Loading model weights took 0.0633 GB
嵌入模型加载成功


In [4]:
# 为文档生成嵌入向量
print("正在生成文档向量...")
doc_texts = [f"passage: {doc}" for doc in page_doc]
doc_outputs = embedding_llm.embed(doc_texts)
doc_embeddings = np.array([output.outputs.embedding for output in doc_outputs])
print(f"文档向量维度: {doc_embeddings.shape}")

正在生成文档向量...


Processed prompts: 100%|██████████| 31/31 [00:00<00:00, 91.82it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

文档向量维度: (31, 384)


In [5]:
# ============================================================
# 第三步：实现简单的向量检索
# ============================================================
def retrieve(query, embeddings, texts, k=3):
    query_output = embedding_llm.embed([f"query: {query}"])
    query_embedding = np.array(query_output[0].outputs.embedding)

    # 计算余弦相似度
    # 先归一化
    query_norm = query_embedding / np.linalg.norm(query_embedding)
    doc_norms = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

    # 点积 = 余弦相似度（已归一化）
    similarities = np.dot(query_norm, doc_norms.T)

    # 取最相似的 k 个
    top_k_indices = np.argsort(similarities)[::-1][:k]
    top_k_docs = [texts[i] for i in top_k_indices]
    top_k_scores = similarities[top_k_indices]

    return top_k_docs, top_k_scores



In [6]:

# ============================================================
# 第四步：加载生成模型
# ============================================================
print("\n正在加载生成模型 Qwen2.5-1.5B-Instruct ...")
gen_llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    gpu_memory_utilization=0.5,
    max_model_len=2048,
    enforce_eager=True
)
print("生成模型加载成功")




正在加载生成模型 Qwen2.5-1.5B-Instruct ...
INFO 07-24 10:31:55 config.py:542] This model supports multiple tasks: {'reward', 'classify', 'embed', 'generate', 'score'}. Defaulting to 'generate'.
WARNING 07-24 10:31:55 cuda.py:95] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 07-24 10:31:55 config.py:678] Async output processing is not supported on the current platform type cuda.
INFO 07-24 10:31:55 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 07-24 10:31:58 model_runner.py:1115] Loading model weights took 2.8873 GB
INFO 07-24 10:31:59 worker.py:267] Memory profiling takes 0.56 seconds
INFO 07-24 10:31:59 worker.py:267] the current vLLM instance can use total_gpu_memory (11.76GiB) x gpu_memory_utilization (0.50) = 5.88GiB
INFO 07-24 10:31:59 worker.py:267] model weights take 2.89GiB; non_torch_memory takes 0.01GiB; PyTorch activation peak memory takes 1.38GiB; the rest of the memory reserved for KV Cache is 1.60GiB.
INFO 07-24 10:31:59 executor_base.py:110] # CUDA blocks: 3747, # CPU blocks: 9362
INFO 07-24 10:31:59 executor_base.py:115] Maximum concurrency for 2048 tokens per request: 29.27x
INFO 07-24 10:32:02 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 3.97 seconds
生成模型加载成功


In [7]:
sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.9,
        max_tokens=256,
        repetition_penalty=1.1
    )

# ============================================================
# 第五步：构建 RAG 问答函数
# ============================================================
def rag_answer(query, k=3):
    """
    检索相关文档，拼接成 prompt，生成答案
    """
    # 检索
    retrieved_docs, scores = retrieve(query, doc_embeddings, page_doc, k=k)

    # 构建 RAG prompt
    context = "\n".join([f"- {doc}" for doc in retrieved_docs])
    rag_prompt = f"""你是一个知识渊博的助手。请根据以下参考资料回答问题。
如果参考资料不足以回答，请如实说明。

【参考资料】
{context}

【问题】
{query}

【回答】"""

    # 生成

    output = gen_llm.generate([rag_prompt], sampling_params)

    return {
        "query": query,
        "retrieved_docs": list(zip(retrieved_docs, scores)),
        "answer": output[0].outputs[0].text
    }

# ============================================================
# 第六步：对比实验 —— 有 RAG vs 无 RAG
# ============================================================
def direct_answer(query):
    """
    不用 RAG，直接问模型
    """
    direct_prompt = f"请回答问题：{query}"
    output = gen_llm.generate([direct_prompt], sampling_params)
    return output[0].outputs[0].text

In [8]:

# ============================================================
# 第七步：运行测试
# ============================================================
test_queries = [
    "胖虎是怎么出现在比奇堡的？",
    "胖虎是怎么打败高达的？",
    "比奇堡都有哪些角色出现了？",
    "谁发明了相对论？",  # 不在知识库中，测试模型是否诚实
]

print("\n" + "="*70)
print("RAG vs 无 RAG 对比实验")
print("="*70)

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"问题: {query}")
    print(f"{'='*70}")

    # RAG 方式
    result = rag_answer(query, k=5)
    print(f"\n[检索到的文档]")
    for doc, score in result["retrieved_docs"]:
        print(f"  [{score:.3f}] {doc}")
    print(f"\n[RAG 回答]")
    print(result["answer"])

    # 直接问答
    direct = direct_answer(query)
    print(f"\n[直接回答]")
    print(direct)

print("\n\n实验完成！请对比以上两种回答的质量和准确性。")


RAG vs 无 RAG 对比实验

问题: 胖虎是怎么出现在比奇堡的？


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.72s/it, est. speed input: 248.40 toks/s, output: 55.85 toks/s]



[检索到的文档]
  [0.906] 胖虎看着屏幕里残破不堪的比奇堡，看着惊慌失措的海底居民，瞬间热血上头。他向来仗义，最喜欢挺身而出保护弱小，当即握紧手中的棒球棍，拍着胸脯大声回应：“别怕！有我胖虎在，没人能破坏你们的家园！我这就过去收拾这台乱搞的机甲！”
  [0.906] 苏醒的高达察觉到了眼前的人类身影，立刻锁定胖虎为新的障碍物。机甲头部的红色传感器亮起刺眼的红光，手臂搭载的光束炮迅速充能，蓝色的能量光波在炮口汇聚，带着极强的破坏力对准胖虎。
  [0.904] 看着家园惨遭破坏，海绵宝宝急得原地转圈，眼泪在眼眶里打转。章鱼哥扔掉竖笛，满脸崩溃，蟹老板死死护住自己的钱袋，却也无力抵挡机甲的碾压。小小的海底居民没有任何对抗巨型机甲的能力，面对庞然大物的高达，所有人都陷入了绝望。
  [0.901] 找准破绽的胖虎深吸一口气，摒弃所有杂念，将全身的力量汇聚于手臂。他不再被动躲闪，主动迎着高达的攻势冲了上去。高达见状，立刻蓄力最强光束炮，湛蓝的巨型能量光束直冲胖虎而来，想要一举击溃这个唯一的阻碍。
  [0.899] 可高达的装甲坚硬无比，胖虎全力一击，只在机甲表面留下一道浅浅的印记，甚至没能撼动机甲分毫。反倒是巨大的反震力，让胖虎手臂发麻，连连后退数步。初次交锋，胖虎落入下风，被高达的绝对战力死死压制。

[RAG 回答]

胖虎是通过自己选择来到比奇堡的。当他在屏幕上看到比奇堡和海里的居民时，他感到了强烈的正义感，并决定挺身而出保护弱小。因此，他握紧手中的棒球棍，向大家宣誓：“别怕！有我胖虎在，没人能破坏你们的家园！我这就过去收拾这台乱搞的机甲！”他的行动体现了他对正义的坚持以及无私的精神。


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.12it/s, est. speed input: 27.61 toks/s, output: 59.46 toks/s]



[直接回答]
 胖虎出现在比奇堡是因为他被一只大狗追着跑，然后在比奇堡迷路了。

问题: 胖虎是怎么打败高达的？


...


[直接回答]
 比奇堡是一部漫画作品，由美国的DC漫画公司创作。它以1960年代的超级英雄比奇堡为背景，讲述了一位名为“比奇堡”的超人和他的朋友们如何对抗邪恶势力的故事。

在漫画中，比奇堡的角色主要包括：

1. 比奇堡（Bizarro）：这是故事的核心角色，他是比奇堡的化身，具有与普通人类相似的身体特征，但拥有超人的能力。
2. 小麦格·安德森（Megans Anderson）：比奇堡的朋友之一，也是他的搭档。她是一名警探兼侦探，擅长破解谜题和解决案件。
3. 乔伊·凯勒姆（Joey Kramer）：另一个重要的朋友，他是一位飞行员，负责保护比奇堡并执行任务。
4. 阿尔文·卡特（Alvin Carter）：一位医生，也是比奇堡的好友，他帮助处理比奇堡的医疗需求。
5. 莱昂纳多·阿姆斯特朗（Leonardo Armstrong）：一名科学家，他开发了比奇堡所需的高科技设备和技术。
6. 约翰·索普（John Spencer

问题: 谁发明了相对论？


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  5.28it/s, est. speed input: 2010.06 toks/s, output: 53.17 toks/s]



[检索到的文档]
  [0.871] 胖虎看着屏幕里残破不堪的比奇堡，看着惊慌失措的海底居民，瞬间热血上头。他向来仗义，最喜欢挺身而出保护弱小，当即握紧手中的棒球棍，拍着胸脯大声回应：“别怕！有我胖虎在，没人能破坏你们的家园！我这就过去收拾这台乱搞的机甲！”
  [0.869] 可高达的装甲坚硬无比，胖虎全力一击，只在机甲表面留下一道浅浅的印记，甚至没能撼动机甲分毫。反倒是巨大的反震力，让胖虎手臂发麻，连连后退数步。初次交锋，胖虎落入下风，被高达的绝对战力死死压制。
  [0.868] 第二章：绝境中的跨次元求助
  [0.867] 话音落下，胖虎一步踏入次元光幕。光影流转之间，他瞬间从陆地穿越到深邃的海底。神奇的次元力量为他附上了海底呼吸的能力，让他无需畏惧海水压力，稳稳地站在比奇堡的沙地之上。看着眼前肆意破坏的高达，胖虎眼神凌厉，周身气场瞬间拉满，一场跨次元的热血对决，正式拉开序幕。
  [0.866] 苏醒的高达察觉到了眼前的人类身影，立刻锁定胖虎为新的障碍物。机甲头部的红色传感器亮起刺眼的红光，手臂搭载的光束炮迅速充能，蓝色的能量光波在炮口汇聚，带着极强的破坏力对准胖虎。

[RAG 回答]

爱因斯坦发明了相对论。


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.52s/it, est. speed input: 6.59 toks/s, output: 59.28 toks/s]


[直接回答]
 相对论是由阿尔伯特·爱因斯坦在1905年提出的一种物理理论。它包括两个部分，一个是狭义相对论（也称为洛伦兹变换），另一个是广义相对论（或称爱因斯坦引力）。这两个理论彻底改变了我们对于时间和空间的理解，并且对现代物理学和天文学产生了深远的影响。

简而言之，爱因斯坦是相对论的创始人。


实验完成！请对比以上两种回答的质量和准确性。
